# 🛡️ NexusReturnGuard — 03 Model
**Goal:** Validate on time split, train LightGBM + CatBoost, tune, final retrain.
**Input:** `full.parquet` from NB02. **Output:** test predictions for NB04.
**Split:** fit month_idx ≤ 18, valid 19–25. Never random.
**Rule:** target encoding built inside each fold, earlier months only.

## Task 1: Load, feature lists, time split, drift check
**Why:** Verify NB02 output, lock split, check suspects (prelaunch, calendar, monthly counts) before modeling.
**Check:** 1,880,000 rows x 104 cols, return rate 0.1805, 0 object cols, test prelaunch ~0.

In [1]:
import os, gc
import pandas as pd, numpy as np

# 1. find + load parquet
PQ = [os.path.join(r, f) for r, _, fs in os.walk('/kaggle/input') for f in fs if f == 'full.parquet'][0]
print(PQ)
full = pd.read_parquet(PQ)
print(full.shape, '| train:', (full.is_train == 1).sum(), '| test:', (full.is_train == 0).sum())
print('train return rate:', full.loc[full.is_train == 1, 'returned'].mean().round(4))

# 2. feature lists
ID_COLS = ['order_line_id','order_id','customer_id','product_id','store_id','supplier_code','banner_id']
NOT_FEATS = ID_COLS + ['order_ts','returned','is_train','month_idx','year']
FEATS = [c for c in full.columns if c not in NOT_FEATS]
CATS = [c for c in FEATS if str(full[c].dtype) == 'category']
print(len(FEATS), 'features |', len(CATS), 'categorical')
print('object cols left:', [c for c in FEATS if full[c].dtype == 'object'])

# 3. time split
train = full[full.is_train == 1]
fit = train[train.month_idx <= 18]
val = train[train.month_idx >= 19]
print('fit', len(fit), fit.returned.mean().round(4), '| val', len(val), val.returned.mean().round(4))

# 4. drift suspects
print(full.groupby('is_train')[['prod_prelaunch','cpi','unemployment','fuel_price','prod_freq_month','cust_order_seq']].mean().round(3))
print(
    train.groupby(
        pd.cut(train.month_idx, [-1, 6, 12, 18, 24, 25]), observed=False
    )['prod_prelaunch']
    .mean()
    .round(3)
)

/kaggle/input/notebooks/preranakamath21/nexusreturnguard-02-features/full.parquet
(1880000, 104) | train: 1400005 | test: 479995
train return rate: 0.1805
92 features | 17 categorical
object cols left: []
fit 1048082 0.1803 | val 351923 0.1811
          prod_prelaunch         cpi  unemployment  fuel_price  \
is_train                                                         
0                  0.000  175.886002         7.258       3.756   
1                  0.216  170.613007         8.167       3.296   

          prod_freq_month  cust_order_seq  
is_train                                   
0                  21.041       64.783997  
1                  17.785       25.122000  
month_idx
(-1, 6]     0.444
(6, 12]     0.286
(12, 18]    0.152
(18, 24]    0.014
(24, 25]    0.000
Name: prod_prelaunch, dtype: float64


## Task 2: Baseline LightGBM
**Why:** Reference AUC for every later change. No target encoding yet.
**Do:** Train on fit (month ≤ 18), early stop on val (19+), print AUC + top-25 gain.
**Look for:** val AUC, fit vs val gap (overfit/time proxy), what the model leans on.

In [2]:
import lightgbm as lgb, time
from sklearn.metrics import roc_auc_score

params = dict(objective='binary', metric='auc', learning_rate=0.05, num_leaves=63,
              min_child_samples=200, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=5, max_bin=127, cat_smooth=20, min_data_per_group=200,
              n_jobs=-1, verbose=-1, seed=42)

def run(feats, params=params, rounds=3000, es=100):
    cats = [c for c in feats if c in CATS]
    dtr = lgb.Dataset(fit[feats], fit['returned'], categorical_feature=cats)
    dva = lgb.Dataset(val[feats], val['returned'], categorical_feature=cats, reference=dtr)
    m = lgb.train(params, dtr, rounds, valid_sets=[dtr, dva], valid_names=['fit', 'val'],
                  callbacks=[lgb.early_stopping(es), lgb.log_evaluation(100)])
    p = m.predict(val[feats], num_iteration=m.best_iteration)
    return m, roc_auc_score(val['returned'], p)

t = time.time()
m0, auc0 = run(FEATS)
print(f"\nBASELINE val AUC {auc0:.4f} | best iter {m0.best_iteration} | fit AUC {m0.best_score['fit']['auc']:.4f} | {time.time()-t:.0f}s")

imp = pd.DataFrame({'feat': FEATS, 'gain': m0.feature_importance('gain')})
imp['gain_pct'] = (imp.gain / imp.gain.sum() * 100).round(2)
print(imp.sort_values('gain', ascending=False).head(25).to_string(index=False))
print('zero-gain feats:', imp.loc[imp.gain == 0, 'feat'].tolist())

Training until validation scores don't improve for 100 rounds
[100]	fit's auc: 0.744657	val's auc: 0.657574
[200]	fit's auc: 0.753916	val's auc: 0.667402
[300]	fit's auc: 0.760654	val's auc: 0.672388
[400]	fit's auc: 0.766474	val's auc: 0.675022
[500]	fit's auc: 0.771886	val's auc: 0.676398
[600]	fit's auc: 0.776812	val's auc: 0.677301
[700]	fit's auc: 0.781685	val's auc: 0.67799
[800]	fit's auc: 0.786465	val's auc: 0.678344
[900]	fit's auc: 0.790972	val's auc: 0.679032
[1000]	fit's auc: 0.79539	val's auc: 0.679385
[1100]	fit's auc: 0.799543	val's auc: 0.67954
[1200]	fit's auc: 0.803731	val's auc: 0.679825
[1300]	fit's auc: 0.807784	val's auc: 0.680134
[1400]	fit's auc: 0.81179	val's auc: 0.680228
[1500]	fit's auc: 0.815651	val's auc: 0.68027
Early stopping, best iteration is:
[1448]	fit's auc: 0.813719	val's auc: 0.680348

BASELINE val AUC 0.6803 | best iter 1448 | fit AUC 0.8137 | 509s
                feat          gain  gain_pct
           promo_num 628510.967976     33.83
  promo_c

## Task 3: Ablation by feature group
**Why:** Fit-val gap is 0.133. Some features memorize the fit period (time proxies). Drop each group, see val AUC.
**Do:** Fast config (lr 0.1), same seed, drop one group at a time + all flagged together.
**Look for:** group where dropping RAISES val AUC = harmful. Delta under 0.001 = noise.

In [3]:
fast = dict(params, learning_rate=0.1, num_leaves=31, max_bin=63)

def run_q(feats, p=fast, rounds=1000, es=40):
    cats = [c for c in feats if c in CATS]
    dtr = lgb.Dataset(fit[feats], fit['returned'], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(val[feats], val['returned'], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(p, dtr, rounds, valid_sets=[dva], valid_names=['val'],
                  callbacks=[lgb.early_stopping(es, verbose=False)])
    pr = m.predict(val[feats], num_iteration=m.best_iteration)
    return m.best_iteration, roc_auc_score(val['returned'], pr)

groups = {
 'trend_calendar': ['cpi', 'unemployment', 'fuel_price'],
 'prelaunch_age':  ['prod_age', 'prod_prelaunch'],
 'cust_history':   ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'],
 'freq_enc':       [c for c in FEATS if c.endswith('_freq') or c == 'prod_freq_month'],
 'age_proxies':    ['store_age', 'cust_tenure'],
 'promo_cat_dup':  ['promo_channel_code'],
 'zero_gain':      ['ord_dup_prod'],
}
groups = {k: [c for c in v if c in FEATS] for k, v in groups.items()}

res = []
t0 = time.time()
it, a = run_q(FEATS); ref = a
res.append(('REFERENCE (all 92)', 0, it, a, 0.0))
print(f"ref {a:.4f} | iter {it} | {time.time()-t0:.0f}s")

for name, drop in groups.items():
    feats = [c for c in FEATS if c not in drop]
    it, a = run_q(feats)
    res.append((f'drop {name}', len(drop), it, a, a - ref))
    print(f"{name:15} drop {len(drop):2} cols | val {a:.4f} | delta {a-ref:+.4f} | iter {it} | {time.time()-t0:.0f}s")

# all flagged together
all_drop = sorted(set(sum(groups.values(), [])))
it, a = run_q([c for c in FEATS if c not in all_drop])
res.append(('drop ALL groups', len(all_drop), it, a, a - ref))
print(f"{'ALL':15} drop {len(all_drop):2} cols | val {a:.4f} | delta {a-ref:+.4f} | iter {it}")

out = pd.DataFrame(res, columns=['run', 'n_dropped', 'best_iter', 'val_auc', 'delta']).round(4)
print(out.sort_values('val_auc', ascending=False).to_string(index=False))

ref 0.6764 | iter 299 | 84s
trend_calendar  drop  3 cols | val 0.6794 | delta +0.0030 | iter 998 | 283s
prelaunch_age   drop  2 cols | val 0.6795 | delta +0.0031 | iter 826 | 456s
cust_history    drop  3 cols | val 0.6762 | delta -0.0001 | iter 519 | 576s
freq_enc        drop  6 cols | val 0.6778 | delta +0.0015 | iter 526 | 695s
age_proxies     drop  2 cols | val 0.6790 | delta +0.0027 | iter 728 | 854s
promo_cat_dup   drop  1 cols | val 0.6514 | delta -0.0250 | iter 3 | 872s
zero_gain       drop  1 cols | val 0.6570 | delta -0.0194 | iter 3 | 891s
ALL             drop 18 cols | val 0.6618 | delta -0.0145 | iter 950
                run  n_dropped  best_iter  val_auc   delta
 drop prelaunch_age          2        826   0.6795  0.0031
drop trend_calendar          3        998   0.6794  0.0030
   drop age_proxies          2        728   0.6790  0.0027
      drop freq_enc          6        526   0.6778  0.0015
 REFERENCE (all 92)          0        299   0.6764  0.0000
  drop cust_history  

## Task 3b: Clean ablation
**Why:** Task 3 reference stopped at iter 299 while others ran to ~1000. Two runs hit an early-stop glitch (iter 3). Unfair.
**Do:** Patience 150 for all runs, REF repeated with seed 7 for noise floor, drop proxy groups cumulatively.
**Look for:** delta above the REF-seed noise = real. Pick the final feature list.

In [4]:
def run_q2(feats, seed=42, rounds=1500, es=150):
    p = dict(fast, seed=seed)
    cats = [c for c in feats if c in CATS]
    dtr = lgb.Dataset(fit[feats], fit['returned'], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(val[feats], val['returned'], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(p, dtr, rounds, valid_sets=[dva], valid_names=['val'],
                  callbacks=[lgb.early_stopping(es, verbose=False)])
    pr = m.predict(val[feats], num_iteration=m.best_iteration)
    return m.best_iteration, roc_auc_score(val['returned'], pr)

P, T, A, F, C = (groups[k] for k in ['prelaunch_age', 'trend_calendar', 'age_proxies', 'freq_enc', 'cust_history'])
BASE = [c for c in FEATS if c != 'ord_dup_prod']          # zero-gain col dropped for good

exps = [
 ('REF seed42',              [],              42),
 ('REF seed7 (noise floor)', [],              7),
 ('drop P+T+A',              P + T + A,       42),
 ('drop P+T+A+F',            P + T + A + F,   42),
 ('drop P+T+A+F+C',          P + T + A + F + C, 42),
 ('drop promo_channel_code', ['promo_channel_code'], 42),
]

res, t0, ref = [], time.time(), None
for name, drop, seed in exps:
    feats = [c for c in BASE if c not in drop]
    it, a = run_q2(feats, seed=seed)
    if ref is None: ref = a
    res.append((name, len(feats), it, round(a, 4), round(a - ref, 4)))
    print(f"{name:26} n_feats {len(feats):3} | iter {it:4} | val {a:.4f} | delta {a-ref:+.4f} | {time.time()-t0:.0f}s")

print(pd.DataFrame(res, columns=['run', 'n_feats', 'best_iter', 'val_auc', 'delta']).to_string(index=False))

REF seed42                 n_feats  91 | iter 1451 | val 0.6801 | delta +0.0000 | 287s
REF seed7 (noise floor)    n_feats  91 | iter 1135 | val 0.6798 | delta -0.0003 | 536s
drop P+T+A                 n_feats  84 | iter 1144 | val 0.6801 | delta -0.0000 | 776s
drop P+T+A+F               n_feats  78 | iter 1014 | val 0.6769 | delta -0.0032 | 986s
drop P+T+A+F+C             n_feats  75 | iter 1432 | val 0.6621 | delta -0.0180 | 1245s
drop promo_channel_code    n_feats  90 | iter 1088 | val 0.6781 | delta -0.0020 | 1484s
                    run  n_feats  best_iter  val_auc   delta
             REF seed42       91       1451   0.6801  0.0000
REF seed7 (noise floor)       91       1135   0.6798 -0.0003
             drop P+T+A       84       1144   0.6801 -0.0000
           drop P+T+A+F       78       1014   0.6769 -0.0032
         drop P+T+A+F+C       75       1432   0.6621 -0.0180
drop promo_channel_code       90       1088   0.6781 -0.0020


## Task 4: Target encoding (store, product, customer, supplier, banner)
**Why:** Id-level return rates were the top single signals (store 0.598, product 0.583, customer 0.544). Raw ids can't go in a tree.
**Do:** Fit rows: expanding rate from EARLIER months only. Val rows: rate from all fit rows. Smoothed with a prior.
**Check:** no label leak. Single TE AUC on val ~0.55–0.60. Model val AUC vs 0.6801 (noise 0.0003).

In [5]:
# 1. integer codes for ids (fast groupby/merge)
TE_KEYS = {'store_id': 50, 'product_id': 20, 'customer_id': 10, 'supplier_code': 50, 'banner_id': 50}  # key: smoothing alpha
for k in TE_KEYS:
    full[k + '_cd'] = pd.factorize(full[k])[0].astype('int32')

train = full[full.is_train == 1]
fit = train[train.month_idx <= 18]
val = train[train.month_idx >= 19]

# 2. target encoding: fit rows = past months only, other rows = all fit rows
def add_te(df_fit, df_oth, keys=TE_KEYS):
    a, b = df_fit.copy(), df_oth.copy()
    prior = a['returned'].mean()
    for k, alpha in keys.items():
        c = k + '_cd'
        g = (a.groupby([c, 'month_idx'])['returned'].agg(s='sum', n='count')
               .reset_index().sort_values([c, 'month_idx']))
        g['ps'] = g.groupby(c)['s'].cumsum() - g['s']      # sum of EARLIER months
        g['pn'] = g.groupby(c)['n'].cumsum() - g['n']      # count of EARLIER months
        a = a.merge(g[[c, 'month_idx', 'ps', 'pn']], on=[c, 'month_idx'], how='left')
        a[k + '_te'] = ((a['ps'] + prior * alpha) / (a['pn'] + alpha)).astype('float32')
        a = a.drop(columns=['ps', 'pn'])
        tot = df_fit.groupby(c)['returned'].agg(s='sum', n='count')
        s = b[c].map(tot['s']).fillna(0); n = b[c].map(tot['n']).fillna(0)
        b[k + '_te'] = ((s + prior * alpha) / (n + alpha)).astype('float32')
    return a, b

fit2, val2 = add_te(fit, val)
assert len(fit2) == len(fit) and len(val2) == len(val)
TE_COLS = [k + '_te' for k in TE_KEYS]
for c in TE_COLS:
    print(f"{c:18} val AUC alone {roc_auc_score(val2['returned'], val2[c]):.4f} | fit mean {fit2[c].mean():.4f} | val mean {val2[c].mean():.4f}")

# 3. model with TE (84 chosen features + 5 TE cols)
FEATS3 = [c for c in BASE if c not in (P + T + A)] + TE_COLS
print(len(FEATS3), 'features')

def run_df(dfit, dval, feats, seed=42, rounds=1500, es=150):
    p = dict(fast, seed=seed)
    cats = [c for c in feats if c in CATS]
    dtr = lgb.Dataset(dfit[feats], dfit['returned'], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(dval[feats], dval['returned'], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(p, dtr, rounds, valid_sets=[dtr, dva], valid_names=['fit', 'val'],
                  callbacks=[lgb.early_stopping(es, verbose=False)])
    pv = m.predict(dval[feats], num_iteration=m.best_iteration)
    return m, m.best_iteration, roc_auc_score(dval['returned'], pv), m.best_score['fit']['auc']

t0 = time.time()
m3, it3, auc3, fauc3 = run_df(fit2, val2, FEATS3)
print(f"\nTE model val AUC {auc3:.4f} | delta vs 0.6801: {auc3-0.6801:+.4f} | iter {it3} | fit AUC {fauc3:.4f} | {time.time()-t0:.0f}s")

imp = pd.Series(m3.feature_importance('gain'), index=FEATS3)
print((imp / imp.sum() * 100).round(2).sort_values(ascending=False).head(15).to_string())

store_id_te        val AUC alone 0.5981 | fit mean 0.1793 | val mean 0.1844
product_id_te      val AUC alone 0.5960 | fit mean 0.1744 | val mean 0.1789
customer_id_te     val AUC alone 0.5670 | fit mean 0.1767 | val mean 0.1757
supplier_code_te   val AUC alone 0.5552 | fit mean 0.1792 | val mean 0.1817
banner_id_te       val AUC alone 0.4955 | fit mean 0.1818 | val mean 0.1813
89 features

TE model val AUC 0.6923 | delta vs 0.6801: +0.0122 | iter 769 | fit AUC 0.7821 | 278s
promo_num             44.45
product_id_te          5.42
store_id_te            3.37
region_match           3.22
promo_channel_code     3.18
customer_type          2.29
weight_kg              1.78
cust_region            1.70
category               1.66
customer_id_te         1.57
freight_ratio          1.48
discount               1.36
freight_cost           1.35
customer_id_freq       1.20
cust_order_seq         1.15


## Task 5: First submission (LightGBM + TE)
**Why:** Bank a score before improving. Check public LB vs val AUC 0.6923.
**Do:** Drop banner_id_te, re-validate, retrain on ALL 24 months with TE from all train, predict test.
**Save:** val_lgb.npy, test_lgb.npy (for blend), submission_lgb.csv.
**Check:** 479,995 rows, probs in [0,1], no NaN, same order as sample_submission.

In [6]:
FEATS4 = [c for c in FEATS3 if c != 'banner_id_te']
print(len(FEATS4), 'features')

# 1. re-validate without banner_id_te + save val preds (for blend later)
t0 = time.time()
m4, it4, auc4, fauc4 = run_df(fit2, val2, FEATS4)
print(f"val AUC {auc4:.4f} | delta vs 0.6923: {auc4-0.6923:+.4f} | iter {it4} | fit {fauc4:.4f} | {time.time()-t0:.0f}s")
val_pred = m4.predict(val2[FEATS4], num_iteration=m4.best_iteration)
np.save('/kaggle/working/val_lgb.npy', val_pred)

# 2. final: TE from ALL train months -> test
train = full[full.is_train == 1]
test  = full[full.is_train == 0]
train2, test2 = add_te(train, test)
assert len(train2) == len(train) and len(test2) == len(test)
print('test TE means:', test2[TE_COLS].mean().round(4).to_dict())

# 3. retrain on all 24 months (rounds = best iter x 1.15, more data)
n_rounds = int(it4 * 1.15)
cats = [c for c in FEATS4 if c in CATS]
dall = lgb.Dataset(train2[FEATS4], train2['returned'], categorical_feature=cats)
mf = lgb.train(dict(fast, seed=42), dall, n_rounds)
test_pred = mf.predict(test2[FEATS4])
np.save('/kaggle/working/test_lgb.npy', test_pred)
print('rounds', n_rounds, '| test pred mean', test_pred.mean().round(4), '| min/max', test_pred.min().round(3), test_pred.max().round(3))

# 4. submission aligned to sample_submission order
SUB = '/kaggle/input/competitions/nexus-opti-ml-2026-operation-insight/sample_submission.csv'
sub = pd.read_csv(SUB)
mp = pd.Series(test_pred, index=test2['order_line_id'].values)
sub['returned'] = sub['order_line_id'].map(mp).values
assert len(sub) == 479_995 and sub['returned'].notna().all() and sub['returned'].between(0, 1).all()
sub.to_csv('/kaggle/working/submission_lgb.csv', index=False)
print(sub.head(), '\nsaved')

88 features
val AUC 0.6903 | delta vs 0.6923: -0.0020 | iter 625 | fit 0.7766 | 237s
test TE means: {'store_id_te': 0.18070000410079956, 'product_id_te': 0.17810000479221344, 'customer_id_te': 0.17569999396800995, 'supplier_code_te': 0.18140000104904175, 'banner_id_te': 0.181099995970726}
rounds 718 | test pred mean 0.1718 | min/max 0.009 0.917
     order_line_id  returned
0  L85d608999fa1ee  0.122030
1  L86272cb9543018  0.307453
2  L66b5669bdace5b  0.099634
3  L2c5886124612de  0.371227
4  L56fe7aa5ac5dd8  0.088878 
saved


## Task 6: Drift diagnostics
**Why:** Top LB is 0.607 but our val is 0.69. Find out if signal decays over time and what looks different in test.
**Do:** (1) val AUC per month, (2) AUC inside each promo group, (3) adversarial check val vs test.
**Look for:** AUC falling month by month, promo carrying the AUC, features that separate val from test.

In [7]:
# 1. val AUC per month (existing predictions, instant)
v = val2[['month_idx', 'returned', 'promo_num']].copy(); v['p'] = val_pred
print('AUC per val month:')
print(
    v.groupby("month_idx")
    .apply(lambda g: roc_auc_score(g.returned, g.p), include_groups=False)
    .round(4)
    .to_string()
)

# 2. how much is promo? AUC of promo alone, and AUC INSIDE each promo group
print('\npromo_num alone AUC:', round(roc_auc_score(v.returned, v.promo_num), 4))
print('AUC inside promo group:')
v.groupby("promo_num").apply(
    lambda g: roc_auc_score(g.returned, g.p), include_groups=False
).round(4).to_string()

# 3. adversarial: can a model tell val rows from test rows?
a = pd.concat([val2.sample(150_000, random_state=1).assign(adv=0),
               test2.sample(150_000, random_state=1).assign(adv=1)])
cut = np.random.RandomState(0).rand(len(a)) < 0.7
ac = [c for c in FEATS4 if c in CATS]
dtr = lgb.Dataset(a.loc[cut, FEATS4], a.loc[cut, 'adv'], categorical_feature=ac)
ma = lgb.train(dict(objective='binary', metric='auc', learning_rate=0.1, num_leaves=31, verbose=-1, n_jobs=-1),
               dtr, 150)
print('\nadversarial AUC (0.5 = same, high = different):',
      round(roc_auc_score(a.loc[~cut, 'adv'], ma.predict(a.loc[~cut, FEATS4])), 4))
imp = pd.Series(ma.feature_importance('gain'), index=FEATS4)
print((imp / imp.sum() * 100).round(1).sort_values(ascending=False).head(10).to_string())

AUC per val month:
month_idx
19    0.7129
20    0.7050
21    0.6888
22    0.6865
23    0.6792
24    0.6684
25    0.5833

promo_num alone AUC: 0.5606
AUC inside promo group:

adversarial AUC (0.5 = same, high = different): 1.0
month                    97.1
cust_days_since_first     2.5
fuel_surcharge            0.3
channel                   0.0
quantity                  0.0
payment_method            0.0
ship_mode                 0.0
unit_price                0.0
line_no                   0.0
items_in_order            0.0


## Task 7: Which features decay over time?
**Why:** Val AUC drops 0.009/month. Find the decaying signals before the final model.
**Do:** (1) single-feature AUC per val month + slope, (2) AUC inside each promo group, (3) corrected adversarial check without time columns.
**Look for:** TE columns with negative slope = stale. Features that separate val from test.

In [8]:
import warnings

# 1. single-feature AUC per val month (19-24) + slope per month
msk = val2.month_idx.between(19, 24).values
v = val2[msk].copy()
v['model'] = val_pred[msk]
cols = [
    'model',
    'promo_num',
    'product_id_te',
    'store_id_te',
    'customer_id_te',
    'supplier_code_te',
    'region_match',
    'weight_kg',
    'items_in_order',
    'quantity',
    'minute1',
    'discount',
    'freight_cost',
    'cust_order_seq',
]


def sauc(g, c):
  x = g[c].fillna(g[c].median())
  a = roc_auc_score(g['returned'], x)
  return max(a, 1 - a)


# Suppress the pandas FutureWarning for this block
with warnings.catch_warnings():
  warnings.simplefilter(action='ignore', category=FutureWarning)
  tab = pd.DataFrame({
      c: v.groupby('month_idx').apply(lambda g, c=c: sauc(g, c)) for c in cols
  }).T

tab['slope_per_mo'] = [
    np.polyfit(range(6), tab.loc[c].values[:6], 1)[0] for c in tab.index
]
print(tab.round(4).sort_values('slope_per_mo').to_string())

# 2. AUC inside each promo group (loop)
print('\nmodel AUC inside promo group:')
for k in sorted(v['promo_num'].dropna().unique()):
  g = v[v.promo_num == k]
  print(int(k), len(g), round(roc_auc_score(g.returned, g.model), 4))

# 3. adversarial again, WITHOUT time columns
TIME = [
    'month',
    'week_of_year',
    'fuel_surcharge',
    'temperature_f',
    'holiday_flag',
    'cust_days_since_first',
    'cust_order_seq',
]
AF = [c for c in FEATS4 if c not in TIME]
a = pd.concat([
    val2.sample(150_000, random_state=1).assign(adv=0),
    test2.sample(150_000, random_state=1).assign(adv=1),
])
cut = np.random.RandomState(0).rand(len(a)) < 0.7
ac = [c for c in AF if c in CATS]
dtr = lgb.Dataset(
    a.loc[cut, AF], a.loc[cut, 'adv'], categorical_feature=ac
)
ma = lgb.train(
    dict(
        objective='binary',
        metric='auc',
        learning_rate=0.1,
        num_leaves=31,
        verbose=-1,
        n_jobs=-1,
    ),
    dtr,
    150,
)
print(
    '\nadversarial AUC without time cols:',
    round(
        roc_auc_score(a.loc[~cut, 'adv'], ma.predict(a.loc[~cut, AF])), 4
    ),
)
imp = pd.Series(ma.feature_importance('gain'), index=AF)
print(
    (imp / imp.sum() * 100)
    .round(1)
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

month_idx             19      20      21      22      23      24  slope_per_mo
promo_num         0.6148  0.5915  0.5701  0.5492  0.5333  0.5056       -0.0212
model             0.7129  0.7050  0.6888  0.6865  0.6792  0.6684       -0.0086
freight_cost      0.5235  0.5273  0.5291  0.5143  0.5151  0.5152       -0.0026
product_id_te     0.6105  0.5998  0.5849  0.5905  0.5953  0.5945       -0.0025
region_match      0.5925  0.5849  0.5647  0.5697  0.5759  0.5832       -0.0020
discount          0.5368  0.5424  0.5471  0.5383  0.5346  0.5338       -0.0013
supplier_code_te  0.5600  0.5561  0.5508  0.5556  0.5553  0.5533       -0.0009
quantity          0.5462  0.5471  0.5365  0.5377  0.5387  0.5454       -0.0008
customer_id_te    0.5667  0.5775  0.5563  0.5599  0.5737  0.5679       -0.0000
minute1           0.5788  0.5752  0.5557  0.5713  0.5781  0.5869        0.0019
weight_kg         0.5532  0.5556  0.5481  0.5575  0.5618  0.5633        0.0023
cust_order_seq    0.5737  0.5883  0.5695  0.5766  0.

## Task 8: Promo decay and no-promo model
**Why:** promo_num AUC falls 0.61 → 0.51 inside val and is 44% of model gain. Test is later still.
**Do:** (1) return rate per promo code per quarter, (2) promo AUC per quarter (signed), (3) model without promo, compared month by month.
**Look for:** did P5's rate collapse? Is the no-promo model flat across months?

In [9]:
# 1. promo return rate per 3-month bucket (all train)
tr_ = full[full.is_train == 1].copy()
tr_['bucket'] = pd.cut(tr_.month_idx, [-1, 3, 6, 9, 12, 15, 18, 21, 25],
                       labels=['1-3', '4-6', '7-9', '10-12', '13-15', '16-18', '19-21', '22-25'])
print(tr_.pivot_table(index='promo_num', columns='bucket', values='returned', aggfunc='mean', observed=True).round(3).to_string())

# 2. promo_num AUC per bucket (signed: below 0.5 = flipped)
print('\npromo_num AUC per bucket (signed):')
print(
    tr_.groupby('bucket', observed=True)
    .apply(
        lambda g: roc_auc_score(g.returned, g.promo_num), include_groups=False
    )
    .round(4)
    .to_string()
)

# 3. model WITHOUT promo, month by month vs the with-promo model
NOPROMO = [c for c in FEATS4 if c not in ('promo_num', 'promo_channel_code')]
t0 = time.time()
mn, itn, aucn, faucn = run_df(fit2, val2, NOPROMO)
print(f"\nNO-PROMO val AUC {aucn:.4f} | iter {itn} | fit {faucn:.4f} | {time.time()-t0:.0f}s")
w = val2[['month_idx', 'returned']].copy()
w['with_promo'] = val_pred
w['no_promo'] = mn.predict(val2[NOPROMO], num_iteration=mn.best_iteration)
w = w[w.month_idx.between(19, 24)]
print(w.groupby('month_idx').apply(
    lambda g: pd.Series({
        'with_promo': roc_auc_score(g.returned, g.with_promo),
        'no_promo':   roc_auc_score(g.returned, g.no_promo)
    }), 
    include_groups=False
).round(4).to_string())

bucket       1-3    4-6    7-9  10-12  13-15  16-18  19-21  22-25
promo_num                                                        
0.0        0.085  0.087  0.089  0.087  0.089  0.105  0.138  0.168
1.0        0.092  0.096  0.093  0.098  0.093  0.111  0.143  0.170
2.0        0.109  0.111  0.108  0.112  0.111  0.127  0.150  0.172
3.0        0.139  0.138  0.138  0.142  0.137  0.148  0.157  0.172
4.0        0.212  0.210  0.207  0.208  0.208  0.203  0.190  0.186
5.0        0.436  0.441  0.446  0.440  0.446  0.391  0.306  0.222

promo_num AUC per bucket (signed):
bucket
1-3      0.7031
4-6      0.7015
7-9      0.7031
10-12    0.6991
13-15    0.7028
16-18    0.6618
19-21    0.5928
22-25    0.5296

NO-PROMO val AUC 0.7070 | iter 613 | fit 0.7457 | 231s
           with_promo  no_promo
month_idx                      
19             0.7129    0.7057
20             0.7050    0.7107
21             0.6888    0.7040
22             0.6865    0.7108
23             0.6792    0.7078
24             0.6684

## Task 9: No-promo model (final)
**Why:** promo signal vanished after month 15. No-promo val 0.7070, flat across months. With-promo 0.6903, decaying.
**Do:** Drop promo_num + promo_channel_code, put banner_id_te back, validate, retrain on all 24 months with 2 seeds, write submission.
**Check:** val AUC ~0.707+, flat per month. Test pred mean ~0.18. 479,995 rows.

In [10]:
# 1. features: no promo, banner TE back
FEATS5 = [c for c in FEATS4 if c not in ('promo_num', 'promo_channel_code')] + ['banner_id_te']
print(len(FEATS5), 'features | promo in list:', [c for c in FEATS5 if 'promo' in c])

# 2. validate + per-month AUC
t0 = time.time()
m5, it5, auc5, fauc5 = run_df(fit2, val2, FEATS5)
print(f"val AUC {auc5:.4f} | vs no-promo 0.7070: {auc5-0.7070:+.4f} | iter {it5} | fit {fauc5:.4f} | {time.time()-t0:.0f}s")
val_pred5 = m5.predict(val2[FEATS5], num_iteration=m5.best_iteration)
np.save('/kaggle/working/val_nopromo.npy', val_pred5)
w = val2[['month_idx', 'returned']].copy(); w['p'] = val_pred5; w = w[w.month_idx.between(19, 24)]
print(
    w.groupby('month_idx')
    .apply(lambda g: roc_auc_score(g.returned, g.p), include_groups=False)
    .round(4)
    .to_string()
)

# 3. final: all 24 months, 2 seeds averaged (train2/test2 from Task 5 already hold TE columns)
n_rounds = int(it5 * 1.15)
cats = [c for c in FEATS5 if c in CATS]
dall = lgb.Dataset(train2[FEATS5], train2['returned'], categorical_feature=cats, free_raw_data=False)
preds = []
for s in [42, 7]:
    mf = lgb.train(dict(fast, seed=s), dall, n_rounds)
    preds.append(mf.predict(test2[FEATS5]))
    print('seed', s, 'done')
print('seed corr:', np.corrcoef(preds[0], preds[1])[0, 1].round(4))
test_pred5 = np.mean(preds, axis=0)
np.save('/kaggle/working/test_nopromo.npy', test_pred5)
print('rounds', n_rounds, '| test pred mean', test_pred5.mean().round(4), '| min/max', test_pred5.min().round(3), test_pred5.max().round(3))

# 4. submission
sub = pd.read_csv(SUB)
mp = pd.Series(test_pred5, index=test2['order_line_id'].values)
sub['returned'] = sub['order_line_id'].map(mp).values
assert len(sub) == 479_995 and sub['returned'].notna().all() and sub['returned'].between(0, 1).all()
sub.to_csv('/kaggle/working/submission_nopromo.csv', index=False)
print(sub.head(), '\nsaved')

87 features | promo in list: []
val AUC 0.7071 | vs no-promo 0.7070: +0.0001 | iter 603 | fit 0.7451 | 223s
month_idx
19    0.7050
20    0.7110
21    0.7045
22    0.7113
23    0.7086
24    0.7036
seed 42 done
seed 7 done
seed corr: 0.9729
rounds 693 | test pred mean 0.1806 | min/max 0.01 0.925
     order_line_id  returned
0  L85d608999fa1ee  0.125589
1  L86272cb9543018  0.383765
2  L66b5669bdace5b  0.090064
3  L2c5886124612de  0.239168
4  L56fe7aa5ac5dd8  0.094171 
saved
